<a href="https://colab.research.google.com/github/anushka28766-beep/Spotify_Catalogue3/blob/main/spotifycatalogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded=files.upload()




In [ ]:
# 1. Check missing values & drop rows missing key info
print("Missing values before cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0])
df = df.dropna(subset=['artists', 'track_name', 'popularity', 'duration_ms']).copy()

# 2. Remove duplicate tracks (keeping highest popularity version)
before_dups = len(df)
df = df.sort_values('popularity', ascending=False).drop_duplicates(
    subset=['track_name', 'artists'], keep='first'
)
print(f"Removed {before_dups - len(df)} duplicate track entries.")

# 3. NumPy feature engineering: duration in minutes & popularity tier
df['duration_min'] = np.round(df['duration_ms'] / 60000, 2)

conditions = [
    df['popularity'] >= 75,
    (df['popularity'] >= 40) & (df['popularity'] < 75),
    df['popularity'] < 40,
]
choices = ['Hit', 'Moderate', 'Niche']
df['popularity_tier'] = np.select(conditions, choices, default='Unknown')

# 4. Count how many artists are credited on each track
df['num_artists'] = df['artists'].astype(str).str.count(';') + 1

print("Cleaned shape:", df.shape)
print("\nPopularity Tier counts:\n", df['popularity_tier'].value_counts())

In [ ]:
# Show up to 10 rows and all columns in output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df.head(10)


In [ ]:
pd.set_option('display.max_rows', 100)  # Allows displaying up to 100 rows
print(df[['artists', 'track_name', 'popularity']].head(100).to_string())

In [ ]:
artist_query = input('Enter an artist name: ').strip()

if not artist_query:
  print('Please enter an artist name.')
else:
  matches = df[
      df['artists']
      .astype(str)
      .str.contains(artist_query, case=False, na=False)
  ].copy()

  if matches.empty:
    print(f"No songs found matching '{artist_query}'.")
  else:
    matches = matches.sort_values('popularity', ascending=False)
    cols = ['track_name', 'artists', 'popularity']
    if 'album_name' in matches.columns:
      cols.append('album_name')
    if 'duration_min' in matches.columns:
      cols.append('duration_min')
    if 'track_genre' in matches.columns:
      cols.append('track_genre')

    print(
        f"\nFound {len(matches)} track(s) matching '{artist_query}' "
        f"(Avg Popularity: {matches['popularity'].mean():.1f}):\n"
    )
    print(matches[cols].head(25).to_string(index=False))